**Dataset**
labeled datasset collected from twitter (Lab 1 - Hate Speech.tsv)

**Objective**
classify tweets containing hate speech from other tweets. <br>
0 -> no hate speech <br>
1 -> contains hate speech <br>

**Total Estimated Time = 90-120 Mins**

**Evaluation metric**
macro f1 score

### Import used libraries

In [98]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
import nltk
from nltk.corpus import stopwords
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 500)
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [99]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Dataset

###### Note: search how to load the data from tsv file

In [100]:
df = pd.read_csv("/content/drive/MyDrive/Lab_1_Hate_Speech.tsv.zip", sep= "\t")
df.head(100)

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,2,0,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,5,0,factsguide: society now #motivation
5,6,0,[2/2] huge fan fare and big talking before they leave. chaos and pay disputes when they get there. #allshowandnogo
6,7,0,@user camping tomorrow @user @user @user @user @user @user @user dannyâ¦
7,8,0,the next school year is the year for exams.ð¯ can't think about that ð­ #school #exams #hate #imagine #actorslife #revolutionschool #girl
8,9,0,we won!!! love the land!!! #allin #cavs #champions #cleveland #clevelandcavaliers â¦
9,10,0,@user @user welcome here ! i'm it's so #gr8 !


### Data splitting

It is a good practice to split the data before EDA helps maintain the integrity of the machine learning process, prevents data leakage, simulates real-world scenarios more accurately, and ensures reliable model performance evaluation on unseen data.

In [101]:
df['label'].value_counts()
#unsampling

,count
label,
0,29322
1,2213


In [102]:
X=df['tweet']
y=df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

### EDA on training data

- check NaNs

In [103]:
df.isna().sum()

,0
id,0
label,0
tweet,0


- check duplicates

In [104]:
df.duplicated().sum()

np.int64(0)

- show a representative sample of data texts to find out required preprocessing steps

In [105]:
df.sample(10)

,id,label,tweet
25863,26291,0,bday to all fathers!! .. we loved you all
30072,30500,0,2 days until graduation #gettingserious #almosthere #countdown #proud #happy then senior week worry begins ð¬ð¬ð¬ð¬
13165,13471,0,our next show is monday at the fiddler's elbow in camden. it's looking like it's going to be a huge night!
20216,20627,0,"taking my parents to see @user tonight,i'm nervous as it's the 1st time i'm leaving my 4 month old son at home ð¥ð©ð­ #coldplaywembley"
3563,3678,0,@user nova and the pursuit of healthy immune system.
30679,31107,0,hanging me upside down ð¤ð #upsidedown #smiles #couple #boyfriend #girlfriend #playingâ¦
21209,21620,0,@user expected live tweets during @user from @user but as long as he's okay i'm good. #loveislove
10280,10417,0,"yooo this bitch needs therapy, dique ""i still want you"" get tf outta here you still need this foot up ya ass ð¯ð smdh"
25243,25671,0,just received a txt to state my bihday celebrations are staing a week early #â¤suprises @user #tophubby long weekend tooð
2619,2661,0,"this #weekend #go far whatever #makes #you , #we are #here to #take #care of #your #laundry. #call..."


In [106]:
df['tweet'].head(5)

,tweet
0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,bihday your majesty
3,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,factsguide: society now #motivation


- check dataset balancing

In [107]:
df['label'].value_counts()

,count
label,
0,29322
1,2213


In [108]:
df['tweet'].head(10)

,tweet
0,@user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction. #run
1,@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx. #disapointed #getthanked
2,bihday your majesty
3,#model i love u take with u all the time in urð±!!! ðððð ð¦ð¦ð¦
4,factsguide: society now #motivation
5,[2/2] huge fan fare and big talking before they leave. chaos and pay disputes when they get there. #allshowandnogo
6,@user camping tomorrow @user @user @user @user @user @user @user dannyâ¦
7,the next school year is the year for exams.ð¯ can't think about that ð­ #school #exams #hate #imagine #actorslife #revolutionschool #girl
8,we won!!! love the land!!! #allin #cavs #champions #cleveland #clevelandcavaliers â¦
9,@user @user welcome here ! i'm it's so #gr8 !


- Cleaning and Preprocessing are:
    - 1
    - 2
    - 3
    - ... etc.

### Cleaning and Preprocessing

#### Extra: use custom scikit-learn Transformers

Using custom transformers in scikit-learn provides flexibility, reusability, and control over the data transformation process, allowing you to seamlessly integrate with scikit-learn's pipelines, enabling you to combine multiple preprocessing steps and modeling into a single workflow. This makes your code more modular, readable, and easier to maintain.

##### link: https://www.andrewvillazon.com/custom-scikit-learn-transformers/

#### Example usage:

In [109]:
from sklearn.base import BaseEstimator, TransformerMixin
import re

class CustomTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
      self.lemmatizer = WordNetLemmatizer()
      self.stop_words = set(stopwords.words('english'))


    def fit(self, X, y=None):
        # Add code for fitting the transformer here
        return self

    def _process_text(self, text):

        #convert small charachters
        text=str(text).lower()

        #remove tags
        text=re.sub(r"http\S+|www\S+|https\S+|@\w+","",text)

        text = re.sub(r'[^\w\s]', '', text)

        text=re.sub(r'[^a-z\s]','',text)

        words = text.split()


        cleane_words=[
            self.lemmatizer.lemmatize(w)
            for w in words if w not in self.stop_words
        ]



        return " ".join(cleane_words)

    def transform(self, X, y=None):
        if isinstance(X, pd.Series):
            return X.apply(self._process_text)
        else:
            return [self._process_text(t) for t in X]

**You  are doing Great so far!**

### Modelling

#### Extra: use scikit-learn pipline

##### link: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

Using pipelines in scikit-learn promotes better code organization, reproducibility, and efficiency in machine learning workflows.

#### Example usage:

In [110]:
from sklearn.pipeline import Pipeline

model = LogisticRegression()

# Create the pipeline
pipeline = Pipeline(steps=[
    ('preprocessing', CustomTransformer()),
    ('Vectorizing', CountVectorizer()),
    ('model', model),
])

#Now you can use the pipeline for training and prediction
pipeline.fit(X_train, y_train)
pipeline.predict(X_test)

array([0, 0, 1, ..., 0, 0, 0])

#### Evaluation

**Evaluation metric:**
macro f1 score

Macro F1 score is a useful metric in scenarios where you want to evaluate the overall performance of a multi-class classification model, **particularly when the classes are imbalanced**

![Calculation](https://assets-global.website-files.com/5d7b77b063a9066d83e1209c/639c3d934e82c1195cdf3c60_macro-f1.webp)

In [111]:
y_pred = pipeline.predict(X_test)
f1_score(y_test, y_pred, average='macro')

0.8155193460447956

In [112]:
test_tweet = ["I am so anger today! #blessed"]
pipeline.predict(test_tweet)

array([0])

### Enhancement

- Using different N-grams
- Using different text representation technique
- Hyperparameter tuning

In [113]:
params={
    "Vectorizing__ngram_range" :[(1,1),(1,2)],
    "Vectorizing__max_features" :[5000,10000],

    'model__C': [0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['liblinear']
}

In [114]:
pipeline = Pipeline(steps=[
    ('preprocessing', CustomTransformer()),
    ('Vectorizing', TfidfVectorizer()),
    ('model', model),
])

In [115]:
grid_search = GridSearchCV(pipeline, params, cv=5, scoring='f1_macro')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing', CustomTransformer()),
                                       ('Vectorizing', TfidfVectorizer()),
                                       ('model', LogisticRegression())]),
             param_grid={'Vectorizing__max_features': [5000, 10000],
                         'Vectorizing__ngram_range': [(1, 1), (1, 2)],
                         'model__C': [0.1, 1, 10], 'model__penalty': ['l2'],
                         'model__solver': ['liblinear']},
             scoring='f1_macro')

### Conclusion and final results


In [116]:
grid_search.best_params_

{'Vectorizing__max_features': 10000,
 'Vectorizing__ngram_range': (1, 2),
 'model__C': 10,
 'model__penalty': 'l2',
 'model__solver': 'liblinear'}

In [117]:
best_model = grid_search.best_estimator_
y_pred2 = best_model.predict(X_test)
f1_score(y_test, y_pred2, average='macro')

0.8377702913139717

#### Done!